In [0]:
productsFilePath = "abfss://miniproject@unluckystorageaccount2.dfs.core.windows.net/products.csv"
display(productsFilePath)

ordersFilePath = "abfss://miniproject@unluckystorageaccount2.dfs.core.windows.net/orders.csv"
display(ordersFilePath)

customersFilePath = "abfss://miniproject@unluckystorageaccount2.dfs.core.windows.net/customers.csv"
display(customersFilePath)

In [0]:
productsDF = spark.read.format("csv").option("header",True).option("inferSchema",True).load(productsFilePath)
#display(productsDF)
ordersDF = spark.read.format("csv").option("header",True).option("inferSchema",True).load(ordersFilePath)
#display(ordersDF)
customersDF = spark.read.format("csv").option("header",True).option("inferSchema",True).load(customersFilePath)
#display(customersDF)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

In [0]:
productsDF.printSchema()

In [0]:

#1.	From products, select only PNAME, CATEGORY, and PRICE.
productsDF1 = productsDF.select("PNAME","CATEGORY","PRICE")
display(productsDF1)
 

In [0]:
#2.	Using col(), create a new column PRICE_WITH_TAX = PRICE * 1.18 (18% GST) on products.
productsDF2 = productsDF1.withColumn("PRICE_WITH_TAX ",col("PRICE") * 1.18)
display(productsDF2)

In [0]:

#3.	Using col(), create a new column DISCOUNTED_PRICE = PRICE * 0.90 (10% discount) on products.
productsDF3 = productsDF2.withColumnRenamed("DISCOUNTED_PRICE", "col(PRICE) * 0.09")

display(productsDF3)

In [0]:
productsDF3.printSchema()

In [0]:
display(productsDF)

In [0]:

#4.	Using selectExpr(), rename PRODID to PRODUCT_ID and PNAME to PRODUCT_NAME, and add a column PRICE_USD = PRICE / 83 (approx INR to USD).
productsDF = productsDF.selectExpr(
    "*",   # keep all existing columns
    "PRODID as PRODUCT_ID",
    "PNAME as PRODUCT_NAME",
    "ROUND(PRICE/83,2) as PRICE_USD"
)
display(productsDF)


In [0]:
customersDF.printSchema()

In [0]:
#5.	Rename the column CNAME to CUSTOMER_NAME in customers using withColumnRenamed().
customersDF = customersDF.withColumnRenamed("CNAME", "CUSTOMER_NAME")
display(customersDF)

In [0]:
#6.	Add two audit columns LOAD_DATE (current_date) and LOAD_TS (current_timestamp) to the orders DataFrame.
customersDF = customersDF.withColumn("LOAD_DATE",current_date()).withColumn("LOAD_TS",current_timestamp())
display(customersDF)

In [0]:
#7.	Drop the columns SUPPLIERID and STOCK from products and display the result.
productsDF = productsDF.drop("SUPPLIERID", "STOCK")
display(productsDF)

In [0]:
#8.	On products, create a new column PRICE_TIER using when()/otherwise() with this logic: PRICE >= 3000 -> 'Premium', PRICE >= 500 -> 'Standard', otherwise 'Budget'.
productsDF = (productsDF.withColumn("PRICE_TIER", when(col("PRICE")> 3000 ,"Premium")
                                                 .when(col("PRICE")>500 ,   "Standard")
                                                 .otherwise("BUDGET")
)
)



In [0]:
display(productsDF)